# HARMONIE to uDALES Radiation Inputs

This notebook documents the recommended workflow for generating `timedepsw.inp.<expnr>` and `timedeplw.inp.<expnr>` for experiment 300 from HARMONIE NWP radiation fields.

The workflow uses:

- `ssrd` for shortwave: accumulated surface solar radiation downwards, converted from J/m2 to W/m2 and mapped to facets.
- `strd` for longwave: accumulated surface thermal radiation downwards, converted from J/m2 to W/m2 and written as scalar sky longwave.

Both scripts average the NWP field over the same central Paris Lambert-93 box used by `grib_to_profile.py`.

## Paths Used Below

These commands assume:

- uDALES repository: `/home/dipanjan/simulation/udtest/u-dales`
- case directory: `/home/dipanjan/simulation/udtest/experiments/300`
- NWP demo directory: `/data/simulateParis/NWP_demo_data`
- HARMONIE version: `8.0`

All executable cells are `%%bash` cells. Run them in order.

## 1. Run Standard uDALES Preprocessing First

This prepares the normal uDALES geometry, IBM, view-factor, SEB, and other input files. Do this before replacing radiation with NWP-derived files.

Important: after generating the NWP radiation files, do not rerun `write_inputs.sh` unless you intend to regenerate or replace radiation inputs again.

In [ ]:
%%bash
set -euo pipefail
cd /home/dipanjan/simulation/udtest
u-dales/tools/write_inputs.sh -p experiments/300


Check the preprocessing log. Re-run this cell while the background preprocessing job is active.

In [ ]:
%%bash
set -euo pipefail
tail -n 80 /home/dipanjan/simulation/udtest/experiments/300/write_inputs.300.log


## 2. Use One Python Environment With Both Dependency Sets

The shortwave full run needs both:

- HARMONIE GRIB dependencies: `xarray`, `cfgrib`, `pyproj`, `eccodes`
- uDALES geometry/radiation dependencies: `trimesh`, `triangle`, `netCDF4`, `directshortwave_f2py`

Prefer the canonical uDALES venv, then add the missing GRIB packages.

In [ ]:
%%bash
set -euo pipefail
cd /home/dipanjan/simulation/udtest/u-dales
tools/python/.venv/bin/python -m pip install cfgrib pyproj eccodes
tools/python/.venv/bin/python - <<'PY'
import xarray
import cfgrib
import pyproj
import trimesh
import triangle
import netCDF4
import udprep.directshortwave_f2py
print('combined radiation environment ok')
PY


## 3. Optional Shortwave Atmospheric Preflight

This reads `ssrd`, converts it to GHI, splits GHI into DNI and diffuse sky irradiance, and stops before loading the STL or tracing facets.

In [ ]:
%%bash
set -euo pipefail
cd /home/dipanjan/simulation/udtest/u-dales
tools/python/.venv/bin/python tools/python/examples/harmonie_to_udales_radiation/harmonie_ssrd_to_timedepsw.py \
  --case-dir /home/dipanjan/simulation/udtest/experiments/300 \
  --atmos-only


For the existing local HARMONIE data, this previously produced 181 uDALES shortwave time points.

## 4. Generate `timedepsw.inp.300`

Run after standard preprocessing has finished and the combined Python environment is active.

In [ ]:
%%bash
set -euo pipefail
cd /home/dipanjan/simulation/udtest/u-dales
tools/python/.venv/bin/python tools/python/examples/harmonie_to_udales_radiation/harmonie_ssrd_to_timedepsw.py \
  --case-dir /home/dipanjan/simulation/udtest/experiments/300 \
  --overwrite


Expected outputs:

- `/home/dipanjan/simulation/udtest/experiments/300/timedepsw.inp.300`
- `/home/dipanjan/simulation/udtest/experiments/300/Sdir.nc`

The `timedepsw` file stores net absorbed shortwave per facet in W/m2. The second line contains 181 times; then one row per facet follows.

## 5. Generate `timedeplw.inp.300`

First do a dry run.

In [ ]:
%%bash
set -euo pipefail
cd /home/dipanjan/simulation/udtest/u-dales
tools/python/.venv/bin/python tools/python/examples/harmonie_to_udales_radiation/harmonie_strd_to_timedeplw.py \
  --case-dir /home/dipanjan/simulation/udtest/experiments/300 \
  --dry-run


Then write the file.

In [ ]:
%%bash
set -euo pipefail
cd /home/dipanjan/simulation/udtest/u-dales
tools/python/.venv/bin/python tools/python/examples/harmonie_to_udales_radiation/harmonie_strd_to_timedeplw.py \
  --case-dir /home/dipanjan/simulation/udtest/experiments/300 \
  --overwrite


Expected output:

- `/home/dipanjan/simulation/udtest/experiments/300/timedeplw.inp.300`

The file has two text header lines followed by 31 rows of model time and `LWsky` in W/m2.

## 6. Quick Output Checks

In [ ]:
%%bash
set -euo pipefail
CASE=/home/dipanjan/simulation/udtest/experiments/300
wc -l "$CASE/timedeplw.inp.300"
head -5 "$CASE/timedeplw.inp.300"
wc -l "$CASE/timedepsw.inp.300"
head -2 "$CASE/timedepsw.inp.300"
ls -lh "$CASE/Sdir.nc"


For experiment 300, `timedeplw.inp.300` should have 33 lines: two headers plus 31 data rows. `timedepsw.inp.300` should have one header, one time row, and one row per facet.

## Notes

- The scripts use local HARMONIE data by default. They do not download data unless `--download` is passed.
- `--download` calls the NWP demo downloader and may download much more than the single radiation field.
- The central Paris domain is fixed by default to EPSG:2154 easting `649375..652600 m` and northing `6861175..6864020 m`.
- Shortwave uses an empirical Erbs direct/diffuse split because the available HARMONIE files include `ssrd` but not a direct-normal or diffuse-only shortwave field.
- Run the NWP radiation scripts after standard preprocessing, not before.